# Lab 07: Creating a Linear Feedback Shift Register

Now that we understand how LFSRs work, let's create one in Python so we can perform Pseudo-random Number Generation (PRNG).

## Review: Bitwise Operations

Recall in Lab 06 that you wrote the following functions to:

* **Get a bit**: Returns the value of a specified bit from a provided number
* **Set a bit**: Returns the value of a number *after* turning a specified bit to 1
* **Clear a bit**: Returns the value of a number *after* turning a specified bit to 0
* **Toggle a bit**: Returns the value of a number *after* flipping a specified bit from 1 to 0, or vice versa

The functions to perform these operations are given below.

<div class="alert alert-warning" role="alert"  style="color: black;">
    
### Binary Notation

Note that these functions work assuming that the right-most bit in a number has an index of 0, not 1. ($b_0$, not $b_1$). This *may* be a change for how we've worked with these numbers previously.

</div>

In [1]:
def get_bit(value, bit_index):
    bitmask = 1 << bit_index
    result = value & bitmask
    return result >> bit_index

In [2]:
def set_bit(value, bit_index):
    bitmask = 1 << bit_index
    result = value | bitmask
    return result 

In [3]:
def clear_bit(value, bit_index):
    bitmask = 1 << bit_index
    result = value & ~bitmask
    return result

In [4]:
def toggle_bit(value, bit_index):
    bitmask = 1 << bit_index
    result = value ^ bitmask
    return result

## LFSR PRNGs

Now that you understand how to shift, get, set, unset, and toggle bit values we know more than enough to write a function that can mimic the LFSR process of generating pseudorandom digits. Let's recall the process of an LFSR, but use the language of shift, get, set, unset, and toggle to describe the operations.

We'll define the LFSR system of having a current state defined by the bits: $b_n, b_{n-1}, ..., b_2, b_1, b_0$ (this is like a row in the table representation that you've seen in class) and an output stream defined by $k_n, k_{n-1}, ..., k_2, k_1, k_0$ (these are the bits that are in the right-most column in the table representation in class).

An LFSR System:
* **gets** the value of the bit at $b_0$ in the current state
* The output stream is **left-shifted** by 1, and the value of the new $k_0$ is either **set** or **unset/cleared** to match $b_0$
* Using the LFSR definition, you **get** the value of some of the bits and **XOR** those values
* The current state is **right-shifted** by 1
* The result of the XOR is used to **set** $b_0$ in the shifted current state

### Question 1

Write a function, `lfsr8` that implements an 8-bit LFSR system with taps at bits 4, 3, 2, and 0. The function should take in a seed value and an integer that represents how many bits the LFSR should output. Use the comments in the scaffolded function below to help you think through what you could do to complete each operation.

In [5]:
def lfsr8(seed, num_bits):
    """
            7 6 5 4 3 2 1 0 (bits == 8)
           ┌─┬─┬─┬─┬─┬─┬─┬─┐
        ┌─→│1│0│1│0│0│0│0│1├─→ output
        │  └─┴─┴─┴┬┴┬┴┬┴─┴┬┘
        └──────XOR┘ │ │   │
                └─XOR─┤   │
                      └XOR┘
                          (taps == 4, 3, 2, 0)
    """
    current_state = seed
    output = 0
    
    for i in range(num_bits):
        
        # gets the value of the bit at $b_0$ in the current state
        output_bit = get_bit(current_state, 0) # SOLUTION
        
        # left-shift the output stream by 1
        shifted_output = output << 1  # SOLUTION
        
        # if the output bit is 1, set k_0 to 1
        if output_bit == 1:
            output = set_bit(shifted_output, 0)  # SOLUTION
        else: #else, just leave k_1 as 0 (assign shifted_output to output "as is")
            output = shifted_output  # SOLUTION
        
        # Get the value of bits 4, 3, 2, and 0 and **XOR** those values to generate a new bit
        new_bit = get_bit(current_state, 4) ^ get_bit(current_state, 3) ^ get_bit(current_state, 2) ^ get_bit(current_state, 0)  # SOLUTION
        
        # Right-shift the current state by 1, creating room for a new bit at index 7
        shifted_current_state = current_state >> 1  # SOLUTION
        
        # if the new_bit is 1, then set b_7 in shifted_current_state to 1 and assign to current_state
        if new_bit == 1:
            current_state = set_bit(shifted_current_state, 7)  # SOLUTION
        else: # otherwise, just leave b_7 as 0 by assigning shifted_current_state to current_state
            current_state = shifted_current_state  # SOLUTION
    
    return output

In [6]:
""" # BEGIN TEST CONFIG
success_message: lfsr8 is returning an integer
failure_message: lfsr8 is not returning an integer
points: 1
""" # END TEST CONFIG

type(lfsr8(0b10100001, 100)) == int

True

In [7]:
""" # BEGIN TEST CONFIG
success_message: lfsr8 is correctly generating 100 bits of PRNG
failure_message: lfsr8 is NOT correctly generating 100 bits of PRNG
points: 1
""" # END TEST CONFIG

lfsr8(0b10100001, 100)

662979776157205528967939712391

In [8]:
""" # BEGIN TEST CONFIG
success_message: lfsr8 correctly returns all 0's when seed is all 0's
failure_message: lfsr8 does not return all 0's when seed is all 0's
points: 1
""" # END TEST CONFIG

lfsr8(0, 50)

0

### Question 2: Turning text into numbers using `ord`

Recall that you can go character by character through a string, and turn that character into a numerical value. Early in the course, we used an alphabet specified by the `LETTERS` string to convert a character into a corresponding number. From now on, we'll start using the `ord` function to perform this step for us.

Run the cell below to see how you can convert a character to an integer using `ord` and then printing out the 8-bit binary version of each character using either `format` (as shown below) or the f-string technique shown in class.

In [9]:
message = "Hello!"

for char in message:
    print( format(ord(char), '08b') )

01001000
01100101
01101100
01101100
01101111
00100001


Write a function `text_to_integer` that accepts a string named `message` and returns a binary representation of `message`. The function should:
* Convert each character, left to right, in `message` to an 8-bit binary string
* Append the 8-bits for that character to the end of a "collection" string
* Once all characters have been appended to the "collection" string, convert it to an integer using the `int` function
* Return the integer version of `message` that you've created


Examples:

```python
>>> text_to_integer("a")
97

>>> text_to_integer("Hi")
18537

>>> text_to_integer("You're so great!")
118880012743380389729479430999113823265
```

In [10]:
def text_to_integer(message):
    # BEGIN SOLUTION NO PROMPT
    binary_message = ''
    
    for char in message:
        binary_message += f"{ord(char):08b}"

    return int(binary_message, 2)
    # END SOLUTION

In [11]:
""" # BEGIN TEST CONFIG
success_message: text_to_integer correctly converts a single character to an integer
failure_message: text_to_integer incorrectly converts a single character to an integer
points: 1
""" # END TEST CONFIG

text_to_integer("a")

97

In [12]:
""" # BEGIN TEST CONFIG
success_message: text_to_integer correctly converts two characters to an integer
failure_message: text_to_integer incorrectly converts two characters to an integer
points: 1
""" # END TEST CONFIG

text_to_integer("Hi")

18537

In [13]:
""" # BEGIN TEST CONFIG
success_message: text_to_integer correctly converts a longer message to an integer
failure_message: text_to_integer incorrectly converts a longer message to an integer
points: 1
""" # END TEST CONFIG

text_to_integer("You're so great!")

118880012743380389729479430999113823265

### Question 3: Encrypt a message

Let's put everything together to encrypt a message!

1. Choose a short string to use as a message. Assign it to `message`.
2. Use the `text_to_integer` function to turn it into an integer assigned to `plaintext`.
3. Use the command `plaintext.bit_length()` to determine how many bits are needed to represent that integer. 
4. Use the `lfsr8` function with a seed your choosing to generate a keystream with the same number of bits as your message assigned to `keystream`
5. Finally, encrypt the message and assign it to `ciphertext`

You've just encrypted a message using your 8-bit LFSR!

In [14]:
message = 'testing' # SOLUTION
plaintext = text_to_integer(message) # SOLUTION
keystream = lfsr8(0b10110111, plaintext.bit_length()) # SOLUTION
ciphertext = plaintext ^ keystream # SOLUTION
ciphertext

721977457661932

# Submitting your work
You're done with this Lab! All assignments in the course will be distributed as notebooks like this one, and you will submit your work by doing the following:
* Save your notebook
* Restart the kernel and run up to this cell.
* Run all the tests by running the cell containing `grader.check_all()`. Make sure they pass the way you expect them to.
* Run the cell below with the code `grader.export(...)`.
* Download the file named `labXX.zip`, found in the explorer pane on the left side of the screen.
* Upload `labXX-<date-time stamp>.zip` to the corresponding lab assignment on Canvas.